# 05 · Engineering Case Studies & Algorithm Benchmarks

**Sistem Fotonika — Digital Fringe Processing**  
**Iwan Cony Setiadi · Teknik Fisika**  
**Level:** S1 semester 6 dengan pengayaan riset · **Mode:** teori + implementasi + eksperimen numerik

> **Status data:** semua citra dan angka dalam notebook ini berasal dari **model/simulasi sintetis** dengan *ground truth*, bukan pengukuran instrumen nyata. Asumsi dan keterbatasan dibahas dalam setiap eksperimen.

**Tujuan:** menyelesaikan tiga studi kasus: deformasi permukaan interferometrik, perpindahan moiré, dan fringe projection 3D yang dikalibrasi secara geometris.

## Rancangan studi kasus


Notebook ini menyatukan tiga alur yang **tidak boleh dipertukarkan rumus fisiknya**:

| Kasus sintetis | Sumber informasi | Measurand | Relasi sensitivitas |
|:---|:---|:---|:---|
| A: refleksi interferometrik | perubahan fase optik bolak-balik | $h(x,y)$ [$\mu$m] | $\Delta\phi=4\pi n h/\lambda_0$ (normal) |
| B: digital moiré in-plane | fase pola kisi referensi/objek | $u_x(x,y)$, $\varepsilon_{xx}$ | $\Delta\phi=-2\pi u_x/p$ |
| C: fringe projection | korespondensi kamera–proyektor | $h(x,y)$ [mm] | geometri stereo: $d=f b/Z$ |

**Validasi:** setiap kasus menggunakan *known ground truth*, skenario noise terdokumentasi, ROI dan RMSE dengan satuan yang benar. Ini adalah **validasi matematis terhadap simulasi**, bukan validasi alat nyata. Bentuk kerja sengaja dibuat bisa diadaptasi menjadi dataset eksperimen setelah tersedia kalibrasi yang sah.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi': 105, 'axes.grid': False,
                     'image.interpolation': 'nearest'})
print('NumPy:', np.__version__)

In [ ]:
from skimage.restoration import unwrap_phase
from scipy.ndimage import gaussian_filter

# Fungsi generik multi-frame phase shifting: I_k = A+B cos(phi+δ_k)
def make_psi_images(phi, offsets, bg=0.5, modulation=0.34,
                    noise_sigma=0.008, seed=1):
    offsets=np.asarray(offsets,dtype=float)
    signal=(np.asarray(bg)+np.asarray(modulation)*
            np.cos(phi[None,:,:]+offsets[:,None,None]))
    return signal+np.random.default_rng(seed).normal(0,noise_sigma,signal.shape)

def four_step_phase(frames):
    assert frames.shape[0]==4
    return np.arctan2(frames[3]-frames[1],frames[0]-frames[2])

def zero_on_reference(recovered,reference):
    return recovered-np.mean(recovered[reference])

def rmse_of(a,b,valid=None):
    err=a-b
    if valid is not None: err=err[valid]
    return float(np.sqrt(np.mean(err**2)))

steps=np.arange(4)*np.pi/2
summary=[]

## A · Interferometri: pengukuran deformasi pelat skala submikrometer


**Skenario:** reflektor mengalami deformasi statis berbentuk kubah. Sensor mensintesis empat citra intensitas pada $\lambda_0=532$ nm. Mengacu pada refleksi normal, rekonstruksi:

$$h(x,y)=\frac{\lambda_0}{4\pi n}[\operatorname{unwrap}(\phi_w(x,y))-\phi_{\mathrm{ref}}].$$

**Asumsi:** indeks bias homogen $n=1$, empat langkah fase diketahui, tidak ada gerakan saat akuisisi, tidak ada polarisasi berbeda, dan wilayah tepi menjadi referensi nol. **Kriteria uji:** RMSE tinggi pada area pengukuran dibanding ground truth relatif terhadap referensi yang sama.

In [ ]:
ny,nx=170,220
Y,X=np.mgrid[:ny,:nx]
Xn=(X-nx/2)/(nx/2); Yn=(Y-ny/2)/(ny/2)
height_true_um=0.27*np.exp(-3.5*(Xn**2+Yn**2))
lam_um=0.532;n_air=1
phi_height=4*np.pi*n_air*height_true_um/lam_um
amplitude=0.36*(1-0.28*np.exp(-((Xn+0.44)**2+(Yn-0.2)**2)/0.07))
framesA=make_psi_images(phi_height,steps,bg=.5,modulation=amplitude,
                        noise_sigma=0.012,seed=104)
wrappedA=four_step_phase(framesA)
unwrappedA=unwrap_phase(wrappedA)
refA=(Xn**2+Yn**2)>0.92
estA=zero_on_reference(unwrappedA,refA)*(lam_um/(4*np.pi*n_air))
trueA=zero_on_reference(phi_height,refA)*(lam_um/(4*np.pi*n_air))
validA=(Xn**2+Yn**2)<0.90
rmseA=rmse_of(estA,trueA,validA)
print(f'A · Height RMSE: {rmseA:.5f} µm')
assert rmseA<0.015
summary.append(('A: interferometri refleksi','RMSE height',rmseA,'µm'))
fig,axs=plt.subplots(1,4,figsize=(14.2,3.1),constrained_layout=True)
for a,v,t in zip(axs,[framesA[0],wrappedA,estA,(estA-trueA)],
                 ['Fringe','Wrapped [rad]','Height [µm]','Residual [µm]']):
    im=a.imshow(v,cmap='gray' if t=='Fringe' else 'viridis')
    a.set_title(t);plt.colorbar(im,ax=a)
plt.show()

### A2 · Mini-riset: pelat lingkaran dijepit dan dibebani tekanan seragam

**Pertanyaan penelitian:** seberapa besar galat rekonstruksi deformasi akibat **ketidakpastian langkah fase** ketika model elastis dan tinggi sejati diketahui?

Berbeda dari contoh kubah Gaussian sebelumnya (hanya bentuk sintetis), target di sini berasal dari teori pelat tipis **Kirchhoff–Love**. Untuk pelat isotropik dengan jari-jari $a$ [m], tebal $t$ [m], modulus Young $E$ [Pa], Poisson $\nu$, tekanan merata $q$ [Pa], tepi dijepit, dan lendutan kecil:

$$D=\frac{Et^3}{12(1-\nu^2)}\quad[\mathrm{N\,m}],\qquad
w(r)=\frac{q}{64D}(a^2-r^2)^2,\qquad 0\leq r\leq a.$$

Pada tepi berlaku $w(a)=w'(a)=0$. Tekanan dipilih agar $w(0)=0{,}27\,\mu$m, **bukan hasil eksperimen**. Karena $w(0)\ll t$, asumsi lendutan kecil konsisten pada contoh ini (perlu diperiksa lagi jika parameter diubah). Fase interferometri refleksi normal: $\phi=4\pi w/\lambda_0$.

Kamera mengambil empat frame $I_k=A+B\cos[\phi+\delta_k]+\eta_k$ tetapi $\delta_k$ sedikit menyimpang dari $k\pi/2$. Bandingkan (i) estimator quadrature yang **mengabaikan** kesalahan langkah dan (ii) *linear least squares* dengan langkah fase **diketahui dari kalibrasi ideal**. Metode (ii) adalah **oracle calibrated benchmark**, bukan algoritma *self-calibration*.

**Variabel kontrol:** $q,E,t,\nu,\lambda_0$, langkah fase dan noise. **Metrik:** RMSE dan bias tinggi [nm], serta peta residu 2D; referensi nol adalah bagian luar pelat. **Uji kegagalan:** perubahan langkah fase, drift antarframe, dan kontras lokal rendah.

In [ ]:
# A2: benchmark mekanika struktur + gangguan phase stepping
E_Pa, nu, t_m, a_m = 70e9, 0.33, 1e-3, 0.020
D_Nm = E_Pa*t_m**3/(12*(1-nu**2))
target_center_m=0.27e-6
pressure_Pa=64*D_Nm*target_center_m/a_m**4
ny2,nx2=172,176
Y2,X2=np.indices((ny2,nx2))
meters_per_pixel=2*a_m/150.0
radius_m=np.hypot(X2-(nx2-1)/2,Y2-(ny2-1)/2)*meters_per_pixel
A2_truth_m=np.where(radius_m<=a_m,pressure_Pa*(a_m*a_m-radius_m*radius_m)**2/(64*D_Nm),0.)
A2_phi=4*np.pi*A2_truth_m/532e-9
actual_steps=steps+np.array([0.,0.09,-0.06,0.13])

def calibrated_ls_phase(stack, actual_offsets):
    """LS per-pixel: I=A+C cos(delta)+S sin(delta); phi=atan2(-S,C)."""
    offsets=np.asarray(actual_offsets)
    design=np.column_stack([np.ones(len(offsets)),np.cos(offsets),np.sin(offsets)])
    beta=np.linalg.lstsq(design,stack.reshape(len(offsets),-1),rcond=None)[0]
    return np.arctan2(-beta[2],beta[1]).reshape(stack.shape[1:])

A2_stack=make_psi_images(A2_phi,actual_steps,noise_sigma=.008,seed=1042)
A2_ref=radius_m>a_m+2*meters_per_pixel
A2_mask=radius_m<a_m-2*meters_per_pixel
A2_quad=zero_on_reference(unwrap_phase(four_step_phase(A2_stack)),A2_ref)*532e-9/(4*np.pi)
A2_ls=zero_on_reference(unwrap_phase(calibrated_ls_phase(A2_stack,actual_steps)),A2_ref)*532e-9/(4*np.pi)
A2_rmse_quad_nm=1e9*rmse_of(A2_quad,A2_truth_m,A2_mask)
A2_rmse_ls_nm=1e9*rmse_of(A2_ls,A2_truth_m,A2_mask)
print(f'A2 · D={D_Nm:.3f} N·m; q={pressure_Pa:.1f} Pa; w(0)={target_center_m*1e9:.1f} nm')
print(f'A2 · 4-step nominal RMSE={A2_rmse_quad_nm:.2f} nm; calibrated-LS RMSE={A2_rmse_ls_nm:.2f} nm')
assert A2_rmse_ls_nm<A2_rmse_quad_nm
assert np.isclose(A2_truth_m.max(),target_center_m,rtol=0.005)
fig,ax=plt.subplots(1,3,figsize=(13,3.7),constrained_layout=True)
for aa,v,title in zip(ax,[A2_truth_m*1e9,A2_quad*1e9-A2_truth_m*1e9,A2_ls*1e9-A2_truth_m*1e9],
                       ['Model pelat: w [nm]','Residu: nominal quadrature [nm]','Residu: LS terkalibrasi [nm]']):
    im=aa.imshow(np.where(radius_m<=a_m,v,np.nan),cmap='viridis')
    aa.set(title=title,xlabel='x kamera [px]',ylabel='y kamera [px]');plt.colorbar(im,ax=aa)
plt.show()

## B · Digital moiré: regangan in-plane dari fase kisi


Ambil pola kisi referensi dengan pitch $p$ piksel dan objek berpindah $u_x(x,y)$ dalam piksel. Kita mensintesis rangkaian citra grating fase-bergeser (untuk referensi **dan** keadaan deformasi):

$$I_{r,k}=A+B\cos\!\left(\frac{2\pi x}{p}+\delta_k\right),\qquad I_{d,k}=A+B\cos\!\left(\frac{2\pi[x-u_x(x,y)]}{p}+\delta_k\right).$$

Setelah estimasi fase dua keadaan, perbedaan fase memenuhi $\Delta\phi=\phi_d-\phi_r=-2\pi u_x/p$. Untuk **small-strain** dan arah koordinat konsisten, $\varepsilon_{xx}=\partial u_x/\partial x$. Konstanta deformasi global tidak terukur tanpa referensi. Di sini dipakai *known reference*, sehingga pergeseran relatif dapat dipulihkan.

**Asumsi:** kisi ideal diketahui, tidak ada occlusion dan perubahan iluminasi non-sinusoidal, fase difference berada dalam rentang tanpa ambiguity global. Estimasi regangan dari turunan spasial harus mempertimbangkan noise dan smoothing.

In [ ]:
ny,nx=168,240
Y,X=np.mgrid[:ny,:nx]
pitch_px=18.0
ux_truth_px=0.28*np.sin(2*np.pi*X/nx)*np.sin(np.pi*Y/ny)
phi_ref=2*np.pi*X/pitch_px
phi_deformed=2*np.pi*(X-ux_truth_px)/pitch_px
ref_stack=make_psi_images(phi_ref,steps,noise_sigma=0.009,seed=201)
def_stack=make_psi_images(phi_deformed,steps,noise_sigma=0.009,seed=202)
ref_phase=four_step_phase(ref_stack)
def_phase=four_step_phase(def_stack)
delta_wrapped=np.angle(np.exp(1j*(def_phase-ref_phase)))
delta_unwrapped=unwrap_phase(delta_wrapped)
ux_est_px=-pitch_px/(2*np.pi)*delta_unwrapped
# Fase absolut terikat pada referensi/deformasi awal, rata-rata global diset 0
ux_est_px-=np.mean(ux_est_px)
ux_truth_rel=ux_truth_px-np.mean(ux_truth_px)
rmseB=rmse_of(ux_est_px,ux_truth_rel)
# strain tak berdimensi = d(u dalam px)/d(x dalam px)
strain_truth=np.gradient(ux_truth_rel,axis=1)
strain_light=np.gradient(gaussian_filter(ux_est_px,sigma=1.2),axis=1)
strain_est=np.gradient(gaussian_filter(ux_est_px,sigma=6.0),axis=1)
strain_rmse_micro=rmse_of(strain_est,strain_truth)*1e6
strain_light_rmse_micro=rmse_of(strain_light,strain_truth)*1e6
print(f'B · Displacement RMSE: {rmseB:.5f} px')
print(f'B · Strain RMSE (Gaussian σ=1.2 px): {strain_light_rmse_micro:.1f} µε')
print(f'B · Strain RMSE (Gaussian σ=6.0 px): {strain_rmse_micro:.1f} µε')
print('Perhatian: smoothing menurunkan noise tetapi mengurangi resolusi spasial strain.')
assert rmseB<0.12
summary.append(('B: digital moiré','RMSE displacement',rmseB,'pixel'))
fig,axs=plt.subplots(1,4,figsize=(14,3.1),constrained_layout=True)
for a,v,t in zip(axs,[def_stack[0],ux_est_px,ux_est_px-ux_truth_rel,strain_est*1e6],
                 ['Fringe deformasi','Displacement [px]','Error [px]','Strain εxx [µε]']):
    im=a.imshow(v,cmap='gray' if t.startswith('Fringe') else 'viridis')
    a.set_title(t);plt.colorbar(im,ax=a)
plt.show()

### B2 · Mini-riset: ambiguitas orde moiré dan dua periode kisi

**Pertanyaan penelitian:** dapatkah perpindahan yang lebih besar daripada setengah periode kisi direkonstruksi dengan benar? Jika hanya satu kisi periodik, informasi perpindahan **ambigu modulo periode** $p$: $u_x$ dan $u_x+m p$ ($m\in\mathbb{Z}$) menghasilkan beda fase terbungkus sama.

Kita simulasikan translasi hampir seragam $u_x\approx 0{,}72p_1$ ditambah deformasi kecil. Estimator satu pitch mengembalikan perpindahan salah orde. Kombinasi dua pitch $p_1\neq p_2$ memberi *synthetic period*:

$$P_{\rm syn}=\frac{1}{\left|1/p_1-1/p_2\right|},\qquad
\Delta\phi_1-\Delta\phi_2=-2\pi u_x\left(\frac1{p_1}-\frac1{p_2}\right).$$

Estimator *coarse* dari beda fase kedua pitch lebih bebas ambiguitas selama rentang pengukuran berada dalam $\pm P_{\rm syn}/2$ (dengan variasi fase spasial cukup halus untuk unwrapping), tetapi noise perpindahannya lebih besar. Gunakan hasil coarse untuk menentukan **bilangan orde integer** dari pitch $p_1$, lalu pertahankan presisi pitch halus.

**Peringatan:** ini adalah *dual-pitch grid phase analysis* sintetis pada satu arah, bukan bukti bahwa semua implementasi geometric moiré dapat melakukan metrologi absolut tanpa kalibrasi atau referensi. Dalam eksperimen nyata, salah identifikasi orde, pola yang tidak sinusoidal, pose kamera, pergeseran rigid-body, dan faktor skala harus diuji.

In [ ]:
# B2: dua periode kisi untuk mengatasi ambiguitas orde global
p1,p2=18.0,23.0
synpitch=1/abs(1/p1-1/p2)
Yb,Xb=np.mgrid[:168,:240]
u_big=0.72*p1+0.75*np.sin(2*np.pi*Xb/240)*np.sin(np.pi*Yb/168)
def phase_difference_for_pitch(pitch,seed):
    reference=2*np.pi*Xb/pitch
    deform=2*np.pi*(Xb-u_big)/pitch
    R=make_psi_images(reference,steps,noise_sigma=.006,seed=seed)
    M=make_psi_images(deform,steps,noise_sigma=.006,seed=seed+100)
    return np.angle(np.exp(1j*(four_step_phase(M)-four_step_phase(R))))

dphi1=phase_difference_for_pitch(p1,620)
dphi2=phase_difference_for_pitch(p2,720)
u_one=-p1/(2*np.pi)*dphi1  # ambigu: interval mendekati [-p1/2,+p1/2)
coarse_phase=np.angle(np.exp(1j*(dphi1-dphi2)))
u_coarse=-synpitch/(2*np.pi)*unwrap_phase(coarse_phase)
integer_order=np.rint((u_coarse-u_one)/p1)
u_refined=u_one+integer_order*p1
rmse_one=rmse_of(u_one,u_big)
rmse_coarse=rmse_of(u_coarse,u_big)
rmse_refined=rmse_of(u_refined,u_big)
print(f'B2 · synthetic pitch={synpitch:.1f} px; true mean displacement={u_big.mean():.2f} px')
print(f'B2 · RMSE single={rmse_one:.3f}, coarse={rmse_coarse:.3f}, refined={rmse_refined:.3f} px')
print('B2 · Persentase orde teridentifikasi benar:',np.mean(np.rint((u_big-u_one)/p1)==integer_order)*100,'%')
assert rmse_refined<rmse_coarse and rmse_refined<0.2
assert rmse_one>5
fig,ax=plt.subplots(1,4,figsize=(15,3.25),constrained_layout=True)
for aa,v,t in zip(ax,[u_big,u_one,u_coarse,u_refined-u_big],
                  ['Ground truth [px]','Satu pitch: ambigu [px]','Dual pitch: coarse [px]','Galat refined [px]']):
    im=aa.imshow(v,cmap='viridis');aa.set(title=t,xlabel='x [px]',ylabel='y [px]');plt.colorbar(im,ax=aa)
plt.show()

## C · Fringe projection profilometry: geometri stereo sederhana


**Peringatan metodologis:** structured-light projection bukan refleksi interferometrik. Karena itu konversi $h=\lambda\phi/(4\pi)$ adalah **salah** untuk studi kasus ini. Ketinggian diperoleh melalui triangulasi dan kalibrasi.

Untuk menunjukkan hubungan geometri secara *derivable*, gunakan model dua pusat optik pinhole dengan sumbu paralel, panjang fokus efektif yang sama $f$ (piksel), baseline $b$ (mm), dan koordinat kamera $u_c$ serta proyektor $u_p$ (piksel):

$$u_c=\frac{fX}{Z},\quad u_p=\frac{f(X-b)}{Z},\quad d=u_c-u_p=\frac{fb}{Z}.$$

Proyektor menampilkan sinusoid berperiode $P$ piksel: $\phi_p=2\pi u_p/P$. Pada piksel kamera sama, dibandingkan permukaan referensi $Z_0$, diperoleh

$$\Delta\phi=\frac{2\pi}{P}(u_p-u_{p,0})=-\frac{2\pi fb}{P}\left(\frac1Z-\frac1{Z_0}\right).$$

Dengan $d_0=fb/Z_0$, rekonstruksi relatif pada model ini adalah

$$\boxed{d=d_0-\frac{P}{2\pi}\Delta\phi,\quad Z=\frac{fb}{d},\quad h=Z_0-Z.}$$

Ini adalah model *rectified, parallel-axis, pinhole* untuk pembelajaran: **belum** mengandung radial distortion, pose ekstrinsik nyata, kalibrasi proyektor, multi-frequency absolute phase, occlusion, maupun pemetaan kamera–proyektor nonideal. Pada sistem nyata, semua itu harus ditangani sebelum mengklaim pengukuran 3D akurat.

In [ ]:
ny,nx=162,224
Y,X=np.mgrid[:ny,:nx]
xc=X-(nx-1)/2
Z0_mm=600.0
f_px=1200.0
b_mm=75.0
P_proj_px=24.0
Xn=(X-(nx-1)/2)/(nx/2)
Yn=(Y-(ny-1)/2)/(ny/2)
h_truth_mm=31.0*np.exp(-3.0*(Xn**2+Yn**2))
Z_truth_mm=Z0_mm-h_truth_mm
up_obj=xc-f_px*b_mm/Z_truth_mm
up_ref=xc-f_px*b_mm/Z0_mm
phase_obj=2*np.pi*up_obj/P_proj_px
phase_ref_fpp=2*np.pi*up_ref/P_proj_px
frames_obj=make_psi_images(phase_obj,steps,noise_sigma=0.014,seed=911)
frames_ref=make_psi_images(phase_ref_fpp,steps,noise_sigma=0.014,seed=912)
wrapped_obj=four_step_phase(frames_obj)
wrapped_ref=four_step_phase(frames_ref)
delta_wrapped=np.angle(np.exp(1j*(wrapped_obj-wrapped_ref)))
delta_cont=unwrap_phase(delta_wrapped)
d0=f_px*b_mm/Z0_mm
d_est=d0-(P_proj_px/(2*np.pi))*delta_cont
assert np.all(d_est>0)
Z_est=f_px*b_mm/d_est
h_est_mm=Z0_mm-Z_est
rmseC=rmse_of(h_est_mm,h_truth_mm)
print('C · Height RMSE:',round(rmseC,4),'mm')
assert rmseC<1.5
summary.append(('C: fringe projection','RMSE height',rmseC,'mm'))
fig,axs=plt.subplots(1,4,figsize=(14.4,3.2),constrained_layout=True)
for a,v,t in zip(axs,[frames_obj[0],delta_cont,h_est_mm,h_est_mm-h_truth_mm],
                 ['Projected fringe (camera)','Δfase ter-unwrapped [rad]',
                  'Rekonstruksi tinggi [mm]','Residual tinggi [mm]']):
    im=a.imshow(v,cmap='gray' if t.startswith('Projected') else 'viridis')
    a.set(title=t,xlabel='x kamera [px]',ylabel='y kamera [px]');plt.colorbar(im,ax=a)
plt.show()

### C2 · Mini-riset: bias gamma dan kesalahan kalibrasi proyektor–kamera

**Pertanyaan penelitian:** apakah menambah jumlah frame selalu memperbaiki pengukuran tinggi? Bagaimana jika proyektor/kamera memiliki respons radiometrik tidak linear atau baseline salah terkalibrasi?

Model intensitas linear $I_{k}=A+B\cos(\phi+2\pi k/N)$ diganggu oleh transformasi respons non-linear $g(I)=I^{\gamma}$, $\gamma=2{,}2$ (contoh fenomenologis, **bukan kalibrasi alat nyata**), lalu noise aditif. Bandingkan $N=4$ vs $N=8$ dengan estimator sinusoidal least-squares. Benchmark mengasumsikan *noise standard deviation* per frame ditingkatkan kira-kira $\sqrt{N/4}$ untuk ilustrasi **anggaran eksposur total tetap**; ini bukan model photon-statistics lengkap.

Selain itu, kita mensimulasikan asumsi baseline $b$ yang terlalu besar 3%. Pada geometri *rectified parallel-pinhole*, estimasi tinggi bergantung pada $f b$, sehingga bias geometri dapat mendominasi sekalipun fase tampak sangat bersih.

**Metrik:** RMSE tinggi [mm], *systematic bias* [mm], dan peta galat. **Eksperimen nyata:** diperlukan kalibrasi intrinsik/ekstrinsik kamera–proyektor, correction distortion, pengujian gamma, dan validasi pada gauge blocks dalam volume kerja.

In [ ]:
# C2: perbandingan estimator 4/8 frame di bawah gamma + satu skenario baseline salah
import pandas as pd

def simulate_fpp(n_steps=4,gamma=1.,baseline_factor=1.,seed=555,base_noise=.010):
    offsets=np.arange(n_steps)*2*np.pi/n_steps
    def synth(phi,seed_offset):
        lin=.50+.34*np.cos(phi[None,:,:]+offsets[:,None,None])
        observed=np.clip(lin,0,1)**gamma
        sig=base_noise*np.sqrt(n_steps/4.)  # noise heuristic; bukan model photon-budget terkalibrasi
        return observed+np.random.default_rng(seed+seed_offset).normal(0,sig,observed.shape)
    I_obj=synth(phase_obj,0)
    I_ref=synth(phase_ref_fpp,15)
    phase_o=calibrated_ls_phase(I_obj,offsets)
    phase_r=calibrated_ls_phase(I_ref,offsets)
    phase_difference=unwrap_phase(np.angle(np.exp(1j*(phase_o-phase_r))))
    baseline_assumed=b_mm*baseline_factor
    k_assumed=f_px*baseline_assumed
    disparity0_assumed=k_assumed/Z0_mm
    d=disparity0_assumed-P_proj_px/(2*np.pi)*phase_difference
    h=Z0_mm-k_assumed/d
    return h,phase_difference

C2_configs=[('4-step linear',4,1.,1.),('4-step gamma',4,2.2,1.),
            ('8-step gamma',8,2.2,1.),('4-step baseline +3%',4,1.,1.03)]
C2_rows=[]
C2_maps=[]
for j,(name,n,gamma,b_factor) in enumerate(C2_configs):
    estimate,_=simulate_fpp(n_steps=n,gamma=gamma,baseline_factor=b_factor,seed=281)
    res=estimate-h_truth_mm
    C2_rows.append({'Eksperimen':name,'RMSE [mm]':rmse_of(estimate,h_truth_mm),
                    'Bias [mm]':float(np.mean(res)), 'P95 |error| [mm]':float(np.percentile(np.abs(res),95))})
    C2_maps.append(res)
C2_table=pd.DataFrame(C2_rows)
print(C2_table.to_string(index=False,float_format=lambda v:f'{v:.4f}'))
assert np.isfinite(C2_table['RMSE [mm]']).all()
fig,ax=plt.subplots(1,4,figsize=(14.6,3.15),constrained_layout=True)
limit=max(np.percentile(np.abs(r),98) for r in C2_maps)
for aa,row,field in zip(ax,C2_rows,C2_maps):
    im=aa.imshow(field,cmap='coolwarm',vmin=-limit,vmax=limit)
    aa.set(title=row['Eksperimen'],xlabel='x kamera [px]',ylabel='y kamera [px]')
    plt.colorbar(im,ax=aa,label='Height residual [mm]')
plt.show()

In [ ]:
# C3: pisahkan bias nonlinearitas dari random noise, seed tidak dipakai saat σ=0
# Desain ini tidak termasuk pengujian lens distortion ataupun pose stereo nyata.
noise_free=[]
for step_count in [4,8]:
    for gamma_value in [1.0,2.2]:
        h_zero,_=simulate_fpp(n_steps=step_count,gamma=gamma_value,
                              baseline_factor=1.,seed=123,base_noise=0.)
        noise_free.append((step_count,gamma_value,rmse_of(h_zero,h_truth_mm),
                           float(np.mean(h_zero-h_truth_mm))))
noise_free_table=pd.DataFrame(noise_free,columns=['N langkah','Gamma','RMSE [mm]','Bias [mm]'])
print('C3 · Noise-free ablation:')
print(noise_free_table.to_string(index=False,float_format=lambda v:f'{v:.6f}'))
assert noise_free_table.loc[noise_free_table['Gamma']==1.,'RMSE [mm]'].max()<1e-6
assert noise_free_table.loc[(noise_free_table['Gamma']==2.2)&(noise_free_table['N langkah']==4),'RMSE [mm]'].iloc[0]>0


**Interpretasi uji deterministik:** dengan noise dihilangkan, perubahan RMSE akibat $\gamma$ tidak bisa disalahartikan sebagai fluktuasi acak. Kinerja 4-step dan 8-step pada nonlinearity tergantung spektrum harmonik respons kamera–proyektor; **tidak ada jaminan bahwa menambah frame selalu lebih baik**, terutama saat gerak objek, read noise, atau perubahan anggaran akuisisi menjadi dominan. Pengujian *time budget* yang sah membutuhkan model shot noise, read noise, dan exposure yang lebih lengkap.

**Interpretasi strain:** turunan spasial memperkuat noise speckle/displacement. Pada contoh sintetis, smoothing Gaussian lemah menghasilkan error regangan besar; smoothing lebih kuat menurunkan varians namun mereduksi detail spasial. Parameter smoothing yang dipilih berdasarkan *ground truth* sintetis tidak boleh langsung dipakai untuk data nyata tanpa validasi dan analisis resolusi spasial.

## 4 · Benchmark dan audit klaim


*Metric* di tiga kasus mempunyai unit **berbeda**, sehingga bukan tiga angka yang bisa diranking mentah-mentah. Nilai RMSE berikut berlaku **hanya untuk parameter simulasi ini**. Untuk membandingkan algoritma secara fair: gunakan measurand, ground truth, noise, ROI, dan photon budget yang identik; laporkan *bias, precision*, batas deteksi, sumber sistematik, runtime, dan failure rate pada beberapa seed.

In [ ]:
import pandas as pd
results=pd.DataFrame(summary,columns=['Kasus','Metrik','RMSE (sintetis)','Satuan'])
print(results.to_string(index=False,float_format=lambda x:f'{x:.5f}'))
assert len(results)==3 and np.isfinite(results['RMSE (sintetis)']).all()

## 4b · Benchmark multi-seed dan ketahanan terhadap noise

Satu RMSE dengan satu seed hanya **demonstrasi**, bukan benchmark yang kuat. Berikut satu *ablation study* otomatis pada ketiga sistem dengan 10 seed dan 3 level noise sintetik per kasus. Ketiga metric tetap berbeda satuannya: **jangan** menjumlah atau membandingkan angka absolut lintas kasus. Ambang gagal di kode adalah *engineering acceptance criteria ilustratif* untuk dataset ini, bukan batas resolusi instrumen nyata. Status gagal unwrapping juga idealnya diperiksa terhadap ground truth dengan kriteria residu $2\pi$.

In [ ]:
noise_settings=[0.004,0.014,0.035]
replicates=10
bench=[]
for noise in noise_settings:
    for seed in range(replicates):
        # A: refleksi interferometrik
        stack=make_psi_images(phi_height,steps,bg=.5,modulation=amplitude,
                              noise_sigma=noise,seed=5000+seed)
        w=four_step_phase(stack)
        hrec=zero_on_reference(unwrap_phase(w),refA)*(lam_um/(4*np.pi*n_air))
        scoreA=rmse_of(hrec,trueA,validA)
        bench.append(('A: interferometri',noise,seed,scoreA,'µm',scoreA>0.015))
        # B: digital moiré; dua rangkaian PSI independen
        R=make_psi_images(phi_ref,steps,noise_sigma=noise,seed=6000+seed)
        D=make_psi_images(phi_deformed,steps,noise_sigma=noise,seed=7000+seed)
        dw=np.angle(np.exp(1j*(four_step_phase(D)-four_step_phase(R))))
        urec=-pitch_px/(2*np.pi)*unwrap_phase(dw)
        urec-=np.mean(urec)
        scoreB=rmse_of(urec,ux_truth_rel)
        bench.append(('B: digital moiré',noise,seed,scoreB,'pixel',scoreB>0.12))
        # C: stereo fringe projection (parameters considered perfectly calibrated)
        O=make_psi_images(phase_obj,steps,noise_sigma=noise,seed=8000+seed)
        R=make_psi_images(phase_ref_fpp,steps,noise_sigma=noise,seed=9000+seed)
        dw=np.angle(np.exp(1j*(four_step_phase(O)-four_step_phase(R))))
        disparity=d0-P_proj_px/(2*np.pi)*unwrap_phase(dw)
        if np.any(disparity<=0):
            scoreC=np.nan;failed=True
        else:
            hrec=Z0_mm-f_px*b_mm/disparity
            scoreC=rmse_of(hrec,h_truth_mm)
            failed=not(np.isfinite(scoreC) and scoreC<1.5)
        bench.append(('C: fringe projection',noise,seed,scoreC,'mm',failed))

bench_df=pd.DataFrame(bench,columns=['Case','Noise sigma','Seed','RMSE','Unit','Failed'])
report=(bench_df.groupby(['Case','Unit','Noise sigma'],as_index=False)
        .agg(median_RMSE=('RMSE','median'),p25_RMSE=('RMSE',lambda z:np.nanpercentile(z,25)),
             p75_RMSE=('RMSE',lambda z:np.nanpercentile(z,75)),
             failures=('Failed','sum'),n=('Seed','count')))
print(report.to_string(index=False,float_format=lambda v:f'{v:.5f}'))
assert len(bench_df)==3*len(noise_settings)*replicates
assert (report['n']==replicates).all()
# Uji tren agregat: high noise tidak seharusnya mengurangi RMSE secara konsisten.
for case in bench_df['Case'].unique():
    a=report[report['Case']==case].sort_values('Noise sigma')['median_RMSE'].values
    assert a[-1]>a[0],f'Check noise trend for {case}'


**Pembacaan hasil:** perubahan median RMSE terhadap noise menunjukkan ketahanan pipeline **pada simulator ini**. Nilai tidak boleh ditafsir sebagai nilai ketidakpastian metrologi hasil pengukuran nyata, sebab kesalahan geometrik, distorsi sensor, speckle decorrelation, drift termal, kalibrasi gerak, maupun anisotropi pixel tidak diikutkan. Untuk publikasi, ulangi dengan dataset eksperimen, model noise yang didukung karakterisasi, dan independent test set.

## 5 · Audit metrologi: sensitivitas, *identifiability*, dan komponen ketidakpastian

Agar studi kasus menjadi latihan riset, mahasiswa harus memisahkan **bias model**, **galat acak**, dan **ambiguity/failure**. Untuk suatu measurand $y=f(\mathbf{x})$ dengan kovarians input $\Sigma_{\mathbf{x}}$, pendekatan orde pertama (GUM):

$$u_c^2(y)\simeq \mathbf{J}\,\Sigma_{\mathbf{x}}\,\mathbf{J}^{\mathsf T},\qquad \mathbf{J}=\partial f/\partial \mathbf{x}.$$

- **A — interferometri:** $h=\lambda_0\Delta\phi/(4\pi n)$, sehingga *phase sensitivity* $\partial h/\partial\phi=\lambda_0/(4\pi n)$. Galat langkah fase tidak sama dengan noise acak piksel.
- **B — moiré/grid:** $u_x=-p\Delta\phi/(2\pi)$, sehingga ketidakpastian $p$ dan skala kamera harus dihitung terpisah; turunan strain memperkuat komponen frekuensi tinggi.
- **C — fringe projection:** untuk $k=fb$ dan $d=d_0-P\Delta\phi/(2\pi)$, $h=Z_0-k/d$. Maka $|\partial h/\partial\Delta\phi|=kP/(2\pi d^2)$; koefisien sensitivitas **bergantung pada kedalaman**.

Aturan laporan: laporkan **unit metrik**, ROI, metode titik nol/referensi, jumlah seed, distribusi error, tingkat kegagalan, dan parameter yang dikalibrasi vs diasumsikan diketahui. **RMSE terhadap ground truth sintetis bukan ketidakpastian pengukuran instrumentasi nyata.**

In [ ]:
# Sensitivitas fase-tegangan displacement-height: tidak bisa dibandingkan tanpa satuan
S_A_nm_per_rad=532/(4*np.pi)     # n=1, λ dalam nm
S_B_px_per_rad=p1/(2*np.pi)
S_C_mm_per_rad=f_px*b_mm*P_proj_px/(2*np.pi*(f_px*b_mm/Z0_mm)**2)
print(f'A: |∂h/∂φ| = {S_A_nm_per_rad:.2f} nm/rad')
print(f'B: |∂ux/∂Δφ| = {S_B_px_per_rad:.2f} pixel/rad (p1={p1} px)')
print(f'C: |∂h/∂Δφ| pada Z0 = {S_C_mm_per_rad:.2f} mm/rad')
# Audit turunan C secara finite-difference pada fase=0
inc=1e-5
K=f_px*b_mm;dzero=K/Z0_mm
f_height=lambda delta:Z0_mm-K/(dzero-P_proj_px*delta/(2*np.pi))
fd=abs((f_height(inc)-f_height(-inc))/(2*inc))
assert abs(fd-S_C_mm_per_rad)/S_C_mm_per_rad<1e-8
print('Audit turunan analitik vs numerik: LULUS')

## 5 · Eksperimen yang membawa tugas ke skala riset


**Ablation study wajib:** jalankan setiap kasus pada sedikitnya 3 tingkat noise dan 10 seed, catat median/IQR RMSE serta jumlah kegagalan unwrapping. Pertanyaan penting adalah: *apakah error didominasi oleh noise, model kalibrasi, atau ambiguitas fase?*

**Penelitian lanjut A:** beam misalignment, phase-step errors, dan drift antarframe pada interferometri; bandingkan estimator Fourier vs 4-step dengan total anggaran cahaya yang sama.

**Penelitian lanjut B:** variasikan pitch grating dan besar regangan; periksa sensitivitas yang meningkat bersamaan dengan aliasing/kontras. Bandingkan estimasi strain menggunakan finite difference vs regularized differentiation.

**Penelitian lanjut C:** ganti geometri paralel ideal dengan kalibrasi kamera–proyektor yang mencakup matriks intrinsik, pose, distorsi, dan *projector pixel correspondence*; uji beda hasil antara model ideal dan kalibrasi penuh.

**Untuk dataset nyata:** simpan metadata instrumen dan kalibrasi, panjang gelombang atau periode proyeksi, exposure/gain, linearitas radiometrik, posisi frame, uncertainty standard, ROI/validity mask, citra mentah asli, skema koordinat, dan pembuktian traceability. Tanpa itu, notebook ini **hanya validasi simulasi**.

## Protokol mini-riset dan output yang dinilai

**Desain eksperimen komputasional wajib:**

1. Tulis hipotesis *falsifiable* untuk **satu variabel gangguan**: A: langkah fase, B: pitch/orde, C: gamma/baseline. Tetapkan model maju, estimator, asumsi, unit, ROI, dan seed **sebelum** melihat hasil.
2. Buat tiga tingkat gangguan, **minimal 10 seed per konfigurasi** sebagai latihan; laporkan median, IQR, bias, RMSE, P95 error dan tingkat gagal. Jalankan *noise-only*, *systematic-only*, dan *combined* sebagai **ablation study**.
3. **Baseline ilmiah:** A: 4-step nominal vs LS calibrated (oracle); B: single-pitch vs dual-pitch coarse vs refined; C: 4/8-step dengan/tanpa gamma dan baseline salah. Seluruh metode dibandingkan pada **measurand dan ROI yang sama** dalam satu kasus.
4. Lakukan *train/validation/test separation*: jika parameter filter atau threshold dipilih dari ground truth pada data validasi, jangan melaporkan skornya lagi sebagai hasil uji independen.
5. Berikan peta *ground truth*, estimasi, residual, profil satu garis, plot error vs gangguan, dan ringkasan *failure cases*. Jangan hanya menampilkan satu citra 3D yang tampak bagus.
6. Rumuskan apa yang **tidak identifiable**: offset fase tanpa referensi, orde integer pada kisi tunggal, dan skala kedalaman jika kalibrasi stereo tidak diketahui.
7. **Transisi dataset nyata:** foto/rekam citra mentah referensi dan deformasi, dark/flat frame, ROI, metadata exposure/gain, parameter geometrik, sertifikat kalibrasi jika ada, temperatur/drift; ukur objek acuan independen. Tanpa tahap ini, hasil tetap **benchmark algoritmik sintetis**.

**Rubrik 100 poin:** ketepatan model fisika 25, derivasi & asumsi 20, implementasi & reproducibility 20, validasi & uncertainty 25, diskusi keterbatasan 10. Tidak ada klaim “siap publikasi” tanpa uji independen dan metode kalibrasi yang sah.

**Pertanyaan HOTS:** (A) kapan empat frame lebih buruk daripada enam jika terjadi drift? (B) mengapa pitch kecil meningkatkan sensitivitas namun mempersulit unwrap? (C) kapan menurunkan phase RMSE tidak memperbaiki depth RMSE akibat baseline error?

## Referensi yang digunakan (terverifikasi metadata)


1. Takeda, M., Ina, H., & Kobayashi, S. (1982). Fourier-transform method of fringe-pattern analysis for computer-based topography and interferometry. *Journal of the Optical Society of America*, 72(1), 156–160. https://doi.org/10.1364/JOSA.72.000156
2. Ghiglia, D. C., & Pritt, M. D. (1998). *Two-Dimensional Phase Unwrapping: Theory, Algorithms, and Software*. Wiley, ISBN 9780471249351. https://www.wiley-vch.de/en/areas-interest/engineering/two-dimensional-phase-unwrapping-978-0-471-24935-1
3. *Calibration of fringe projection profilometry: A comparative review* (2021), *Optics and Lasers in Engineering*. https://doi.org/10.1016/j.optlaseng.2021.106622
4. *Modeling the measurement precision of Fringe Projection Profilometry* (2023), *Light: Science & Applications*. https://www.nature.com/articles/s41377-023-01294-0
5. NIST TN 1297, law of propagation of uncertainty. https://www.nist.gov/pml/nist-technical-note-1297/nist-tn-1297-appendix-law-propagation-uncertainty

Artikel dapat memiliki keterbatasan akses penuh; metadata, abstrak, dan informasi relevan pada laman penerbit telah diperiksa. Tidak ada grafik pada notebook ini yang diklaim sebagai data penelitian eksperimen.

## Literatur dasar dan pembahasan metodologi (tambahan)

- Grédiac, M., Sur, F., & Blaysat, B. (2016), *The Grid Method for In-plane Displacement and Strain Measurement: A Review and Analysis*, *Strain*, 52(3), 205–243. https://doi.org/10.1111/str.12182 — hubungan resolusi spasial, noise, dan bias grid metrology.
- *Calibration of fringe projection profilometry: A comparative review* (2021), *Optics and Lasers in Engineering*. https://doi.org/10.1016/j.optlaseng.2021.106622 — kalibrasi adalah bagian dari model pengukuran, bukan hanya postprocessing.
- Xing & Guo (2018), *Correction of projector nonlinearity in multi-frequency phase-shifting fringe projection profilometry*, *Optics Express*, 26, 16277–16291. https://doi.org/10.1364/OE.26.016277 — artefak fase akibat nonlinearitas radiometrik.
- NIST TN 1297, *Law of Propagation of Uncertainty*. https://www.nist.gov/pml/nist-technical-note-1297/nist-tn-1297-appendix-law-propagation-uncertainty — peran sensitivity coefficient dan kovarians input.

**Batas evidensi:** referensi ini mendasari metodologi; seluruh nilai numerik dalam notebook tetap berasal dari simulasi yang ditentukan di sini, bukan replikasi data artikel.

<!-- AUDIT-RESEARCH-2026 -->

## D · Ekstensi riset singkat: moiré geometris vs korelasi laser speckle

Ketiga kasus A–C mengolah **fase pola periodik**, sedangkan korelasi speckle memanfaatkan **pergeseran tekstur intensitas acak**. Keduanya harus dibedakan: *geometric moiré* dapat terbentuk dari superposisi/multiplikasi dua kisi tanpa interferensi koheren, sementara fully-developed laser speckle muncul dari interferensi medan terhambur koheren.

**Eksperimen D1: geometric moiré.** Dua kisi $g_j(x)=\frac12[1+\cos(2\pi x/p_j)]$ dengan $p_1=13$ dan $p_2=14$ piksel dimultiplikasi. Suku frekuensi selisih dalam produk memiliki periode $P_m=|1/p_1-1/p_2|^{-1}=182$ piksel. Low-pass sengaja memisahkan selubung moiré dari garis kisi berfrekuensi tinggi. Periode diukur melalui puncak spektral; error terbatas resolusi bin DFT dan edge effects.

**Eksperimen D2: laser speckle correlation**. Medan kompleks Gaussian acak yang dihaluskan membentuk intensitas speckle yang mendekati statistik eksponensial di ROI. Speckle digeser secara periodik tepat integer-pixel; fase korelasi FFT dipakai mengukur translasi. *Decorrelation* dimodelkan sebagai campuran **medan** terstruktur dan medan acak independen, bukan tambahan pola moiré. Ini demonstrasi korelasi speckle, **bukan ESPI**; ESPI memerlukan interferensi dengan reference beam dan informasi fase optik terkalibrasi.

Keduanya dites menggunakan data sintetis dan tidak membuktikan sensitivitas absolut instrumen. Ekstensi ini mempertahankan 5 file notebook, dan memisahkan mekanisme pencitraan yang sering tertukar di kelas.

In [ ]:
# D1 -- true geometric moiré (intensity / transmissivity product; no laser required)
from scipy.ndimage import gaussian_filter as gf
L=2048
xx_g=np.arange(L)
p_g1,p_g2=13.,14.
moire_period=1/abs(1/p_g1-1/p_g2)
g1=.5+.5*np.cos(2*np.pi*xx_g/p_g1)
g2=.5+.5*np.cos(2*np.pi*xx_g/p_g2)
product=g1*g2
lowpass=gf(product,sigma=8)
spec=np.abs(np.fft.rfft(lowpass-lowpass.mean()))
freq=np.fft.rfftfreq(L)
# Peak among low-frequency bins only, excluding DC
selected=(freq>0.002)&(freq<0.015)
frequency_peak=freq[selected][np.argmax(spec[selected])]
period_est=1/frequency_peak
print(f'D1 moiré period expected={moire_period:.1f} px, FFT-estimated={period_est:.1f} px')
assert abs(period_est/moire_period-1)<0.06
fig,axs=plt.subplots(1,2,figsize=(12,3.2),constrained_layout=True)
axs[0].plot(xx_g[:600],product[:600],label='Grating product',alpha=.45)
axs[0].plot(xx_g[:600],lowpass[:600],label='Lowpass envelope')
axs[0].set(xlabel='x [px]',ylabel='Transmitansi relatif',title='Geometric moiré — bukan interferometri');axs[0].legend()
axs[1].plot(freq,spec)
axs[1].axvline(1/moire_period,color='black',ls='--',label='Truth beat frequency')
axs[1].set(xlim=(0,.02),xlabel='Spatial frequency [cycles/px]',ylabel='|FFT|',title='Low-frequency moiré peak');axs[1].legend()
plt.show()

# D2 -- coherent complex-field model, FFT cross-correlation, known integer translation
from scipy.ndimage import gaussian_filter
rng_s=np.random.default_rng(584)
ny_s,nx_s=160,192
def random_speckle_field(rng):
    real=gaussian_filter(rng.normal(size=(ny_s,nx_s)),sigma=1.2)
    imag=gaussian_filter(rng.normal(size=(ny_s,nx_s)),sigma=1.2)
    return real+1j*imag
E_ref=random_speckle_field(rng_s)
I_ref_sp=np.abs(E_ref)**2
contrast_sp=float(I_ref_sp.std()/I_ref_sp.mean())
true_shift=(3,-6) # shift applied to deformed image relative to reference (dy,dx)
E_shift=np.roll(E_ref,shift=true_shift,axis=(0,1))
def fft_corr_estimate(a,b):
    F_a=np.fft.fft2(a-a.mean());F_b=np.fft.fft2(b-b.mean())
    cross=np.fft.ifft2(F_b*np.conj(F_a)).real
    idx=np.unravel_index(np.argmax(cross),cross.shape)
    signed=tuple(int(v) if v<=n//2 else int(v-n) for v,n in zip(idx,cross.shape))
    return signed,float(cross.max()/np.sqrt((a-a.mean()).var()*(b-b.mean()).var())/a.size)
E_independent=random_speckle_field(rng_s)
results_s=[]
for rho in [1.0,.85,.55,.25,0.]:
    E_scene=np.sqrt(rho)*E_shift+np.sqrt(1-rho)*E_independent
    I_scene=np.abs(E_scene)**2
    shift_hat,peak_norm=fft_corr_estimate(I_ref_sp,I_scene)
    results_s.append((rho,shift_hat,peak_norm))
print('D2 speckle contrast=',round(contrast_sp,3),'(fully developed ideal expectation ~1)')
print('D2 (coherent field retention, measured displacement, normalized correlation peak):')
for row in results_s: print(row)
assert 0.75<contrast_sp<1.25
assert results_s[0][1]==true_shift
assert results_s[-1][2] < results_s[0][2]
fig,ax=plt.subplots(1,3,figsize=(12.5,3.5),constrained_layout=True)
ax[0].imshow(I_ref_sp,cmap='gray');ax[0].set_title('Speckle reference')
ax[1].imshow(np.abs(E_shift)**2,cmap='gray');ax[1].set_title(f'Speckle translated {true_shift} px')
ax[2].plot([r[0] for r in results_s],[r[2] for r in results_s],'o-')
ax[2].set(xlabel='Coherent field retention ρ',ylabel='Normalized correlation peak',title='Sensitivity to decorrelation')
plt.show()